In [ ]:
import pandas as pd
import geopandas as gpd
import ast
import numpy as np

In [ ]:
fire_obs = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region_w_country.shp')

In [ ]:
res = []

for reg in ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]:

    for season in ["all_seasons", "MAM", "JJA", "SON", "DJF"]:

        sync_eve = pd.read_csv(f'/net/rain/hyclimm/data/projects/SynFire/WP1/Analyze_synchronized_event_pair/Synchronized_event_pair_between_region_pair_{season}.csv')

        sync_eve_reg = sync_eve[(sync_eve['reg1'] == reg)|(sync_eve['reg2'] == reg)]

        n = len(sync_eve_reg)

        if n == 0:
            
            res.append({'reg': reg, 
                        'season': season, 
                        'multi_country_fraction': np.nan})
        else:
        
            multi_ctr_counter = 0
            
            for _, row in sync_eve_reg.iterrows():
            
                if row['reg1'] == reg:
                    
                    eve_reg = row['id1']
                    
                else:
                    
                    eve_reg = row['id2']
            
                eve_reg = ast.literal_eval(eve_reg)
            
                if len(eve_reg) > 1:
                    
                    contri_list = [fire_obs[fire_obs['ptch_id'] == eve]['country'].item() for eve in eve_reg]
                    contri_list_unq = set(contri_list)
            
                    if len(contri_list_unq) > 1:
                        
                        multi_ctr_counter += 1
    
            res.append({'reg': reg, 
                        'season': season, 
                        'multi_country_fraction': multi_ctr_counter/n})

In [ ]:
res_df = pd.DataFrame(res)

In [ ]:
res_df.to_csv('/net/rain/hyclimm/data/projects/SynFire/WP1/multi_country_event_fraction/multi_country_event_fraction_2001_2020.csv', index = False)